In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack"
)

OUTPUT_DIR = DATA_DIR.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Notebook 02 started")
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)

Notebook 02 started
Data: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack
Output: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs


In [2]:
patients = pd.read_csv(DATA_DIR / "patient_360_reference.csv")
tele = pd.read_csv(DATA_DIR / "teleconsultations.csv")
outcomes = pd.read_csv(DATA_DIR / "episode_outcomes.csv")

ncd = pd.read_csv(DATA_DIR / "ncd_screening.csv")
prescriptions = pd.read_csv(DATA_DIR / "prescriptions.csv")
dispensing = pd.read_csv(DATA_DIR / "medicine_dispensing.csv")
labs = pd.read_csv(DATA_DIR / "lab_tests.csv")
followups = pd.read_csv(DATA_DIR / "followup_visits.csv")
visits = pd.read_csv(DATA_DIR / "visit_history.csv")
outreach = pd.read_csv(DATA_DIR / "outreach_actions.csv")

print("Datasets loaded")
print("Patients:", patients.shape)
print("Teleconsultations:", tele.shape)
print("Outcomes:", outcomes.shape)
print("NCD:", ncd.shape)
print("Prescriptions:", prescriptions.shape)
print("Dispensing:", dispensing.shape)
print("Labs:", labs.shape)
print("Followups:", followups.shape)
print("Visits:", visits.shape)
print("Outreach:", outreach.shape)

Datasets loaded
Patients: (5000, 14)
Teleconsultations: (5516, 23)
Outcomes: (5516, 6)
NCD: (4748, 18)
Prescriptions: (5151, 14)
Dispensing: (3756, 14)
Labs: (2005, 13)
Followups: (2575, 11)
Visits: (13061, 11)
Outreach: (6088, 12)


In [3]:
linkage = pd.read_csv(
    OUTPUT_DIR / "teleconsultation_linkage.csv"
)

print("Linkage shape:", linkage.shape)

display(linkage.head())

Linkage shape: (1635, 5)


,source_system,patient_name,predicted_patient_id,confidence,candidate_score
0,teleconsultations,Mohan Laxman,P000004,HIGH,98.0
1,teleconsultations,Mohan Laxman,P000004,HIGH,98.0
2,teleconsultations,Anitha Narsimha,P000007,HIGH,100.0
3,teleconsultations,Anitha Narsimha,P000007,HIGH,100.0
4,teleconsultations,Ramesh Begum,P000008,HIGH,98.0


In [5]:
# CELL 4 — Build exact teleconsultation → patient mapping

import re

def normalize_text(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]", "", value)
    return value


# Normalize identity fields
tele["name_norm"] = tele["patient_name"].apply(normalize_text)
tele["village_norm"] = tele["village"].apply(normalize_text)

patients_temp = patients.copy()

patients_temp["name_norm"] = (
    patients_temp["canonical_name"]
    .apply(normalize_text)
)

patients_temp["village_norm"] = (
    patients_temp["village"]
    .apply(normalize_text)
)


# Exact name + village matching
exact_map = tele.merge(
    patients_temp[
        [
            "patient_id",
            "name_norm",
            "village_norm"
        ]
    ],
    on=[
        "name_norm",
        "village_norm"
    ],
    how="inner"
)


exact_map = exact_map[
    [
        "teleconsult_id",
        "patient_id"
    ]
].drop_duplicates()


print("Exact teleconsultation matches:", len(exact_map))
print("Total teleconsultations:", len(tele))

display(exact_map.head(10))

Exact teleconsultation matches: 5429
Total teleconsultations: 5516


,teleconsult_id,patient_id
0,TC0000001,P000003
1,TC0000002,P000004
2,TC0000002,P003002
3,TC0000003,P000004
4,TC0000003,P003002
5,TC0000004,P000005
6,TC0000005,P000006
7,TC0000006,P000007
8,TC0000006,P001800
9,TC0000007,P000007


In [ ]:
# CELL 5 — Load linkage output from Notebook 01

linkage_path = OUTPUT_DIR / "teleconsultation_linkage.csv"

linkage = pd.read_csv(linkage_path)

print("Linkage loaded:", linkage.shape)
display(linkage.head())

Exact teleconsultation matches: 5429
Total teleconsultations: 5516


,teleconsult_id,patient_id
0,TC0000001,P000003
1,TC0000002,P000004
2,TC0000002,P003002
3,TC0000003,P000004
4,TC0000003,P003002
5,TC0000004,P000005
6,TC0000005,P000006
7,TC0000006,P000007
8,TC0000006,P001800
9,TC0000007,P000007


In [7]:
# CELL 5 — Load linkage output from Notebook 01

linkage_path = OUTPUT_DIR / "teleconsultation_linkage.csv"

linkage = pd.read_csv(linkage_path)

print("Linkage loaded:", linkage.shape)
display(linkage.head())

Linkage loaded: (1635, 5)


,source_system,patient_name,predicted_patient_id,confidence,candidate_score
0,teleconsultations,Mohan Laxman,P000004,HIGH,98.0
1,teleconsultations,Mohan Laxman,P000004,HIGH,98.0
2,teleconsultations,Anitha Narsimha,P000007,HIGH,100.0
3,teleconsultations,Anitha Narsimha,P000007,HIGH,100.0
4,teleconsultations,Ramesh Begum,P000008,HIGH,98.0


In [8]:
# CELL 6 — Inspect linkage

print("Columns:")
print(linkage.columns.tolist())

print("\nRows:", len(linkage))
print("\nConfidence:")
print(linkage["confidence"].value_counts(dropna=False))

Columns:
['source_system', 'patient_name', 'predicted_patient_id', 'confidence', 'candidate_score']

Rows: 1635

Confidence:
confidence
HIGH      1291
MEDIUM     341
LOW          3
Name: count, dtype: int64


In [11]:
linkage = pd.read_csv(
    OUTPUT_DIR / "teleconsultation_linkage.csv"
)

print("Linkage:", linkage.shape)
print(linkage.columns.tolist())

display(linkage.head())

Linkage: (1635, 6)
['source_system', 'teleconsult_id', 'patient_name', 'predicted_patient_id', 'confidence', 'candidate_score']


,source_system,teleconsult_id,patient_name,predicted_patient_id,confidence,candidate_score
0,teleconsultations,TC0000002,Mohan Laxman,P000004,HIGH,98.0
1,teleconsultations,TC0000003,Mohan Laxman,P000004,HIGH,98.0
2,teleconsultations,TC0000006,Anitha Narsimha,P000007,HIGH,100.0
3,teleconsultations,TC0000007,Anitha Narsimha,P000007,HIGH,100.0
4,teleconsultations,TC0000008,Ramesh Begum,P000008,HIGH,98.0


In [12]:
# CELL 7 — Combine exact + fuzzy linkage

fuzzy_map = linkage[
    [
        "teleconsult_id",
        "predicted_patient_id"
    ]
].copy()

fuzzy_map.rename(
    columns={
        "predicted_patient_id": "patient_id"
    },
    inplace=True
)

exact_map_clean = exact_map[
    [
        "teleconsult_id",
        "patient_id"
    ]
].copy()

tele_patient_map = pd.concat(
    [
        exact_map_clean,
        fuzzy_map
    ],
    ignore_index=True
)

tele_patient_map = tele_patient_map.drop_duplicates(
    subset="teleconsult_id",
    keep="first"
)

print("Mapped teleconsultations:", len(tele_patient_map))
print("Total teleconsultations:", len(tele))
print(
    "Unmapped:",
    len(tele) - len(tele_patient_map)
)

Mapped teleconsultations: 5516
Total teleconsultations: 5516
Unmapped: 0


In [13]:
# CELL 8 — Build episode-level foundation

tele_clean = tele.drop(
    columns=["name_norm", "village_norm"],
    errors="ignore"
)

episode_base = (
    tele_clean
    .merge(
        tele_patient_map,
        on="teleconsult_id",
        how="left"
    )
    .merge(
        outcomes,
        on="teleconsult_id",
        how="left",
        suffixes=("", "_outcome")
    )
)

print("Episode base:", episode_base.shape)

display(
    episode_base[
        [
            "episode_id",
            "teleconsult_id",
            "consult_date",
            "patient_id",
            "cohort",
            "lost_to_followup_label",
            "dropout_stage_label"
        ]
    ].head(10)
)

Episode base: (5516, 29)


,episode_id,teleconsult_id,consult_date,patient_id,cohort,lost_to_followup_label,dropout_stage_label
0,E0000001,TC0000001,2026-05-19,P000003,DEVELOPMENT,0.0,Completed care journey
1,E0000002,TC0000002,2026-07-22,P000004,EVALUATION,NaN,NaN
2,E0000003,TC0000003,2026-08-11,P000004,EVALUATION,NaN,NaN
3,E0000004,TC0000004,2026-06-02,P000005,DEVELOPMENT,0.0,Completed care journey
4,E0000005,TC0000005,2026-04-24,P000006,DEVELOPMENT,1.0,Medicine not collected
5,E0000006,TC0000006,2026-02-26,P000007,DEVELOPMENT,1.0,Review not attended
6,E0000007,TC0000007,2026-03-17,P000007,DEVELOPMENT,1.0,Medicine not collected
7,E0000008,TC0000008,2026-02-06,P000008,DEVELOPMENT,0.0,Completed care journey
8,E0000009,TC0000009,2026-07-19,P000008,EVALUATION,NaN,NaN
9,E0000010,TC0000010,2026-03-23,P000009,DEVELOPMENT,1.0,Medicine not collected


In [14]:
# CELL 9 — Linkage coverage

total = len(episode_base)

mapped = episode_base["patient_id"].notna().sum()

unmapped = total - mapped

print("Total episodes:", total)
print("Mapped:", mapped)
print("Unmapped:", unmapped)
print(f"Mapping coverage: {mapped / total:.2%}")

Total episodes: 5516
Mapped: 5516
Unmapped: 0
Mapping coverage: 100.00%


In [15]:
# CELL 10 — Development cohort baseline

development = episode_base[
    episode_base["cohort"] == "DEVELOPMENT"
].copy()

completed = (
    development["lost_to_followup_label"] == 0
).sum()

ltfu = (
    development["lost_to_followup_label"] == 1
).sum()

print("Development episodes:", len(development))
print("Completed:", completed)
print("LTFU:", ltfu)
print(f"LTFU rate: {ltfu / len(development):.2%}")

Development episodes: 4132
Completed: 2144
LTFU: 1988
LTFU rate: 48.11%


In [16]:
# CELL 11 — Dropout stage distribution

stage_counts = (
    development[
        development["dropout_stage_label"].notna()
    ]["dropout_stage_label"]
    .value_counts()
)

display(stage_counts)

stage_percentage = (
    stage_counts / stage_counts.sum() * 100
).round(2)

display(
    stage_percentage.rename("percentage")
)

dropout_stage_label
Completed care journey    2144
Medicine not collected    1170
Review not attended        543
Test not completed         275
Name: count, dtype: int64

dropout_stage_label
Completed care journey    51.89
Medicine not collected    28.32
Review not attended       13.14
Test not completed         6.66
Name: percentage, dtype: float64

In [17]:
# CELL 12 — LTFU by consultation mode

mode_analysis = (
    development
    .groupby("consult_mode")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

mode_analysis["ltfu_rate"] = (
    mode_analysis["ltfu_rate"] * 100
).round(2)

display(mode_analysis)

,consult_mode,episodes,ltfu_rate
0,Assisted video,1396,47.64
1,Audio,1379,48.08
2,Video,1357,48.64


In [18]:
# CELL 13 — LTFU by connectivity quality

connectivity_analysis = (
    development
    .groupby("connectivity_quality")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

connectivity_analysis["ltfu_rate"] = (
    connectivity_analysis["ltfu_rate"] * 100
).round(2)

display(connectivity_analysis)

,connectivity_quality,episodes,ltfu_rate
0,Fair,1441,48.44
1,Good,1372,48.25
2,Poor,1319,47.61


In [19]:
# CELL 14 — LTFU by distance to facility

development["distance_band"] = pd.cut(
    development["distance_to_facility_km"],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=[
        "<=2 km",
        "2-5 km",
        "5-10 km",
        ">10 km"
    ]
)

distance_analysis = (
    development
    .groupby("distance_band", observed=False)
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

distance_analysis["ltfu_rate"] = (
    distance_analysis["ltfu_rate"] * 100
).round(2)

display(distance_analysis)

,distance_band,episodes,ltfu_rate
0,<=2 km,45,20.00
1,2-5 km,211,34.60
2,5-10 km,592,37.50
3,>10 km,3284,51.28


In [20]:
# CELL 15 — LTFU by care requirement

for column in [
    "medicine_advised",
    "test_advised",
    "review_advised"
]:
    
    result = (
        development
        .groupby(column)
        .agg(
            episodes=("episode_id", "count"),
            ltfu_rate=("lost_to_followup_label", "mean")
        )
        .reset_index()
    )

    result["ltfu_rate"] = (
        result["ltfu_rate"] * 100
    ).round(2)

    print(f"\n===== {column} =====")
    display(result)


===== medicine_advised =====


,medicine_advised,episodes,ltfu_rate
0,No,968,27.79
1,Yes,3164,54.33



===== test_advised =====


,test_advised,episodes,ltfu_rate
0,No,2649,42.62
1,Yes,1483,57.92



===== review_advised =====


,review_advised,episodes,ltfu_rate
0,No,1323,35.53
1,Yes,2809,54.04


In [21]:
# CELL 16 — Add canonical patient context

patient_context = patients[
    [
        "patient_id",
        "age_as_of_2026",
        "gender",
        "village",
        "block",
        "district",
        "preferred_language",
        "known_ncd_status",
        "vulnerability_group",
        "household_id"
    ]
].copy()

development_360 = development.merge(
    patient_context,
    on="patient_id",
    how="left",
    suffixes=("", "_patient")
)

print("Patient 360 shape:", development_360.shape)

display(
    development_360[
        [
            "episode_id",
            "patient_id",
            "age_as_of_2026",
            "gender_patient",
            "known_ncd_status",
            "vulnerability_group",
            "lost_to_followup_label"
        ]
    ].head(10)
)

Patient 360 shape: (4132, 39)


,episode_id,patient_id,age_as_of_2026,gender_patient,known_ncd_status,vulnerability_group,lost_to_followup_label
0,E0000001,P000003,47,M,NaN,Low digital access,0.0
1,E0000004,P000005,46,M,Hypertension,Elderly living alone,0.0
2,E0000005,P000006,18,M,Hypertension,Seasonal migrant,1.0
3,E0000006,P000007,18,F,Hypertension+Diabetes,Elderly living alone,1.0
4,E0000007,P000007,18,F,Hypertension+Diabetes,Elderly living alone,1.0
5,E0000008,P000008,66,M,NaN,General,0.0
6,E0000010,P000009,43,M,Hypertension+Diabetes,Low digital access,1.0
7,E0000011,P000009,43,M,Hypertension+Diabetes,Low digital access,0.0
8,E0000012,P000010,61,F,NaN,General,0.0
9,E0000013,P000010,61,F,NaN,General,0.0


In [22]:
# CELL 17 — Vulnerability analysis

vulnerability_analysis = (
    development_360
    .groupby("vulnerability_group")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

vulnerability_analysis["ltfu_rate"] = (
    vulnerability_analysis["ltfu_rate"] * 100
).round(2)

display(vulnerability_analysis)

,vulnerability_group,episodes,ltfu_rate
0,Elderly living alone,276,51.09
1,General,2920,47.02
2,Low digital access,493,50.10
3,Mobility limitation,246,47.56
4,Seasonal migrant,197,55.84


In [23]:
# CELL 18 — NCD analysis

ncd_analysis = (
    development_360
    .groupby("known_ncd_status")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

ncd_analysis["ltfu_rate"] = (
    ncd_analysis["ltfu_rate"] * 100
).round(2)

display(ncd_analysis)

,known_ncd_status,episodes,ltfu_rate
0,Diabetes,737,48.44
1,Hypertension,1072,47.20
2,Hypertension+Diabetes,546,53.85


In [24]:
# CELL 19 — Save Patient 360

patient360_path = OUTPUT_DIR / "patient360_development.csv"

development_360.to_csv(
    patient360_path,
    index=False
)

print("Patient 360 saved:", patient360_path)
print("Rows:", len(development_360))
print("Columns:", len(development_360.columns))

Patient 360 saved: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs\patient360_development.csv
Rows: 4132
Columns: 39


In [25]:
# CELL 20 — Save EDA outputs

mode_analysis.to_csv(
    OUTPUT_DIR / "eda_consult_mode.csv",
    index=False
)

connectivity_analysis.to_csv(
    OUTPUT_DIR / "eda_connectivity.csv",
    index=False
)

distance_analysis.to_csv(
    OUTPUT_DIR / "eda_distance.csv",
    index=False
)

vulnerability_analysis.to_csv(
    OUTPUT_DIR / "eda_vulnerability.csv",
    index=False
)

ncd_analysis.to_csv(
    OUTPUT_DIR / "eda_ncd.csv",
    index=False
)

stage_counts.to_csv(
    OUTPUT_DIR / "eda_dropout_stage.csv"
)

print("EDA outputs saved.")

EDA outputs saved.


In [26]:
# CELL 21 — Age-band analysis

development_360["age_band"] = pd.cut(
    development_360["age_as_of_2026"],
    bins=[0, 18, 30, 45, 60, 75, 120],
    labels=[
        "<=18",
        "19-30",
        "31-45",
        "46-60",
        "61-75",
        "76+"
    ]
)

age_analysis = (
    development_360
    .groupby("age_band", observed=False)
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

age_analysis["ltfu_rate"] = (
    age_analysis["ltfu_rate"] * 100
).round(2)

display(age_analysis)

,age_band,episodes,ltfu_rate
0,<=18,157,44.59
1,19-30,384,46.61
2,31-45,1205,47.80
3,46-60,1376,48.33
4,61-75,806,48.26
5,76+,204,53.43


In [27]:
# CELL 22 — Preferred language analysis

language_analysis = (
    development_360
    .groupby("preferred_language")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

language_analysis["ltfu_rate"] = (
    language_analysis["ltfu_rate"] * 100
).round(2)

display(language_analysis)

,preferred_language,episodes,ltfu_rate
0,English,188,53.19
1,Telugu,3541,47.84
2,Urdu,403,48.14


In [28]:
# CELL 23 — Gender analysis

gender_analysis = (
    development_360
    .groupby("gender_patient")
    .agg(
        episodes=("episode_id", "count"),
        ltfu_rate=("lost_to_followup_label", "mean")
    )
    .reset_index()
)

gender_analysis["ltfu_rate"] = (
    gender_analysis["ltfu_rate"] * 100
).round(2)

display(gender_analysis)

,gender_patient,episodes,ltfu_rate
0,F,2170,46.96
1,M,1962,49.39


In [29]:
# CELL 24 — Save remaining EDA outputs

age_analysis.to_csv(
    OUTPUT_DIR / "eda_age.csv",
    index=False
)

language_analysis.to_csv(
    OUTPUT_DIR / "eda_language.csv",
    index=False
)

gender_analysis.to_csv(
    OUTPUT_DIR / "eda_gender.csv",
    index=False
)

development_360.to_csv(
    OUTPUT_DIR / "patient360_development.csv",
    index=False
)

print("Notebook 02 outputs saved successfully.")

Notebook 02 outputs saved successfully.


In [30]:
# CELL 25 — Notebook 02 final check

required_outputs = [
    "patient360_development.csv",
    "eda_dropout_stage.csv",
    "eda_distance.csv",
    "eda_connectivity.csv",
    "eda_consult_mode.csv",
    "eda_medicine_advised.csv",
    "eda_test_advised.csv",
    "eda_review_advised.csv",
    "eda_vulnerability.csv",
    "eda_ncd.csv",
    "eda_age.csv",
    "eda_language.csv",
    "eda_gender.csv"
]

print("Checking outputs...\n")

for filename in required_outputs:
    path = OUTPUT_DIR / filename
    print(
        "OK  " if path.exists() else "MISS ",
        filename
    )

Checking outputs...

OK   patient360_development.csv
OK   eda_dropout_stage.csv
OK   eda_distance.csv
OK   eda_connectivity.csv
OK   eda_consult_mode.csv
MISS  eda_medicine_advised.csv
MISS  eda_test_advised.csv
MISS  eda_review_advised.csv
OK   eda_vulnerability.csv
OK   eda_ncd.csv
OK   eda_age.csv
OK   eda_language.csv
OK   eda_gender.csv
